In [4]:
import pandas as pd
import boto3
import io

# Configurações
arquivo_csv = "consumo_energia_limpo.csv"
bucket = "desafio-da-sprint04"
pasta_s3 = "analises-etapa2/"
perfil_aws = "Yanmaiscedo"

# Leitura do arquivo diretamente do Bucker
session = boto3.Session(profile_name=perfil_aws)
s3 = session.client("s3")

obj = s3.get_object(Bucket=bucket, Key=f"{arquivo_csv}")
df = pd.read_csv(io.BytesIO(obj["Body"].read()), parse_dates=["mes_ano_corrigido"])

# Analise 1: Função de conversão + subtração direta entre colunas numéricas.
# Comparação entre consumo atual e média histórica
df['diferenca'] = df['consumo_mes_referencia'] - df['media_consumo_mes_2018_2019']
analise1 = df[['sigla_orgao', 'media_consumo_mes_2018_2019', 'consumo_mes_referencia', 'diferenca']].head(10)
analise1_txt = "analise1_comparacao_consumo.txt"
analise1.to_string(open(analise1_txt, "w", encoding="utf-8"), index=False)

# Analise 2: Função condicional + Agregação simples
# Quantidade de órgãos que atingiram a meta de economia
df['atingiu_meta'] = df['consumo_mes_referencia'] < df['media_consumo_mes_2018_2019']
resumo_meta = df['atingiu_meta'].value_counts().rename(index={True: 'Atingiram a meta', False: 'Não atingiram'})
analise2_txt = "analise2_resumo_metas.txt"
with open(analise2_txt, "w", encoding="utf-8") as f:
    f.write("Resumo de metas atingidas:\n")
    f.write(resumo_meta.to_string())

# Analise 3: Filtro com operador lógico  + Agregação por grupo
# Top 5 maiores consumidores em 2022
df['ano'] = df['mes_ano_corrigido'].dt.year
consumo_2022 = df[df['ano'] == 2022]
top_consumidores = consumo_2022.groupby('sigla_orgao')['consumo_mes_referencia'].sum().sort_values(ascending=False).head(5)
analise3_txt = "analise3_top_consumidores_2022.txt"
top_consumidores.to_string(open(analise3_txt, "w", encoding="utf-8"))

# Analise 4: Função de string + Organização temporal
# Evolução mensal do consumo de órgãos da área de saúde
min_saude = df[df['orgao'].str.contains('saúde', case=False, na=False)]
evolucao_saude = min_saude[['mes_ano_corrigido', 'consumo_mes_referencia']].sort_values('mes_ano_corrigido')
analise4_txt = "analise4_evolucao_min_saude.txt"
evolucao_saude.to_string(open(analise4_txt, "w", encoding="utf-8"), index=False)

# Analise 5: Função de data + Função condicional  + Agregação temporal
# Total de consumo por semestre
df['semestre'] = df['mes_ano_corrigido'].dt.month.apply(lambda m: 1 if m <= 6 else 2)
consumo_semestral = df.groupby(['ano', 'semestre'])['consumo_mes_referencia'].sum().sort_index()
analise5_txt = "analise5_consumo_semestral.txt"
consumo_semestral.to_string(open(analise5_txt, "w", encoding="utf-8"))

# Envio dos arquivos para o Bucket
for arquivo in [analise1_txt, analise2_txt, analise3_txt, analise4_txt, analise5_txt]:
    s3.upload_file(arquivo, bucket, f"{pasta_s3}{arquivo}")
    print(f" Enviado: {arquivo} → s3://{bucket}/{pasta_s3}{arquivo}")

 Enviado: analise1_comparacao_consumo.txt → s3://desafio-da-sprint04/analises-etapa2/analise1_comparacao_consumo.txt
 Enviado: analise2_resumo_metas.txt → s3://desafio-da-sprint04/analises-etapa2/analise2_resumo_metas.txt
 Enviado: analise3_top_consumidores_2022.txt → s3://desafio-da-sprint04/analises-etapa2/analise3_top_consumidores_2022.txt
 Enviado: analise4_evolucao_min_saude.txt → s3://desafio-da-sprint04/analises-etapa2/analise4_evolucao_min_saude.txt
 Enviado: analise5_consumo_semestral.txt → s3://desafio-da-sprint04/analises-etapa2/analise5_consumo_semestral.txt
